# Sentiment Analysis of IMDb Movie Reviews Using Apache Spark and PySpark Machine Learning

**Course:** DASC6021 Big Data Analytics & Data Visualization  
**Tools:** Apache Spark, PySpark MLlib, Jupyter Notebook  
**Dataset:** `data/imdb_reviews.csv`

This notebook implements a scalable sentiment analysis pipeline for IMDb movie reviews. It loads a CSV dataset into a Spark DataFrame, cleans and tokenizes text, removes stopwords, extracts TF-IDF features, trains Logistic Regression, Naive Bayes, and Decision Tree classifiers, and evaluates them using accuracy, precision, recall, and F1-score.


## 1. Environment Setup

The notebook assumes that Java, Apache Spark, PySpark, and Jupyter are installed. The accompanying `README.md` explains installation steps. Run all cells from top to bottom after placing `imdb_reviews.csv` in the `data/` folder.


In [1]:
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, NaiveBayes, DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF

spark = (
    SparkSession.builder
    .appName("IMDb Sentiment Analysis with PySpark")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


Spark version: 3.5.1


## 2. Load the IMDb Reviews Dataset

The expected input file is `data/imdb_reviews.csv`. Common IMDb review files use columns such as `review` and `sentiment`, but the loader below also accepts close alternatives such as `text`, `label`, or `class`.


In [2]:
DATA_PATH = Path("data/imdb_reviews.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Place the IMDb dataset at data/imdb_reviews.csv. "
        "Expected columns are review/text and sentiment/label."
    )

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(str(DATA_PATH))
)

print("Rows:", raw_df.count())
raw_df.printSchema()
raw_df.show(5, truncate=80)


Rows: 50000
root
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)
+--------------------------------------------------------------------------------+---------+
|                                                                          review|sentiment|
+--------------------------------------------------------------------------------+---------+
|One of the other reviewers has mentioned that after watching just 1 Oz episod...| positive|
|A wonderful little production. <br /><br />The filming technique is very unass...| positive|
|I thought this was a wonderful way to spend time on a too hot summer weekend...| positive|
|Basically there's a family where a little boy (Jake) thinks there's a zombie i...| negative|
|Petter Mattei's "Love in the Time of Money" is a visually stunning film to w...| positive|
+--------------------------------------------------------------------------------+---------+
only showing top 5 rows


## 3. Standardize Columns and Inspect Class Balance

This step renames the text and target columns to `review` and `label`. Positive sentiment is encoded as `1.0`; negative sentiment is encoded as `0.0`.


In [3]:
columns_lower = {c.lower(): c for c in raw_df.columns}

text_candidates = ["review", "text", "content", "comment"]
label_candidates = ["sentiment", "label", "class", "target"]

text_col = next((columns_lower[c] for c in text_candidates if c in columns_lower), None)
label_col = next((columns_lower[c] for c in label_candidates if c in columns_lower), None)

if text_col is None or label_col is None:
    raise ValueError(
        f"Could not identify text and label columns. Available columns: {raw_df.columns}"
    )

df = raw_df.select(
    F.col(text_col).cast("string").alias("review"),
    F.col(label_col).alias("raw_label")
).dropna()

df = df.withColumn(
    "label",
    F.when(F.lower(F.col("raw_label").cast("string")).isin("positive", "pos", "1", "true"), F.lit(1.0))
     .when(F.lower(F.col("raw_label").cast("string")).isin("negative", "neg", "0", "false"), F.lit(0.0))
     .otherwise(F.col("raw_label").cast(DoubleType()))
).dropna(subset=["label"])

df = df.select("review", "label")
df.cache()

print("Clean rows:", df.count())
df.groupBy("label").count().orderBy("label").show()


Clean rows: 50000
+-----+-----+
|label|count|
+-----+-----+
|  0.0|25000|
|  1.0|25000|
+-----+-----+


In [4]:
balance_pdf = df.groupBy("label").count().orderBy("label").toPandas()
balance_pdf["sentiment"] = balance_pdf["label"].map({0.0: "Negative", 1.0: "Positive"})

plt.figure(figsize=(6, 4))
plt.bar(balance_pdf["sentiment"], balance_pdf["count"], color=["#D1495B", "#2A9D8F"])
plt.title("IMDb Review Class Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of reviews")
plt.tight_layout()
plt.show()


## 4. Text Preprocessing

Reviews often contain HTML tags, punctuation, numbers, mixed case, and extra spaces. The preprocessing below lowercases reviews, removes HTML tags, removes non-letter characters, and collapses whitespace. Tokenization and stopword removal are performed inside Spark ML pipeline stages so the same transformations are applied consistently during training and testing.


In [5]:
clean_df = (
    df.withColumn("clean_review", F.lower(F.col("review")))
      .withColumn("clean_review", F.regexp_replace("clean_review", r"<[^>]+>", " "))
      .withColumn("clean_review", F.regexp_replace("clean_review", r"[^a-zA-Z\s]", " "))
      .withColumn("clean_review", F.regexp_replace("clean_review", r"\s+", " "))
      .withColumn("clean_review", F.trim(F.col("clean_review")))
      .filter(F.length("clean_review") > 0)
)

clean_df.select("review", "clean_review", "label").show(3, truncate=100)


+--------------------+--------------------+-----+
|              review|        clean_review|label|
+--------------------+--------------------+-----+
|One of the other ...|one of the other ...|  1.0|
|A wonderful littl...|a wonderful littl...|  1.0|
|I thought this wa...|i thought this wa...|  1.0|
+--------------------+--------------------+-----+
only showing top 3 rows


In [6]:
length_df = clean_df.withColumn("word_count", F.size(F.split(F.col("clean_review"), " ")))
length_pdf = length_df.select("word_count").sample(False, 0.2, seed=42).toPandas()

plt.figure(figsize=(8, 4))
plt.hist(length_pdf["word_count"], bins=40, color="#4E79A7", edgecolor="white")
plt.title("Distribution of Review Lengths")
plt.xlabel("Words per review")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 5. TF-IDF Feature Extraction

The model pipeline uses:

1. `RegexTokenizer` to split cleaned reviews into words.
2. `StopWordsRemover` to remove common English stopwords.
3. `CountVectorizer` to create term-frequency vectors.
4. `IDF` to down-weight very common terms and produce TF-IDF features.


In [7]:
train_df, test_df = clean_df.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

tokenizer = RegexTokenizer(
    inputCol="clean_review",
    outputCol="tokens",
    pattern=r"\W+",
    minTokenLength=2
)

stopwords = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

count_vectorizer = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="tf_features",
    vocabSize=50000,
    minDF=2.0
)

idf = IDF(
    inputCol="tf_features",
    outputCol="features"
)


Training rows: 39981
Testing rows: 10019


## 6. Train PySpark ML Classification Models

Three supervised classifiers are trained for comparison:

- Logistic Regression
- Naive Bayes
- Decision Tree


In [8]:
models = {
    "Logistic Regression": LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=50,
        regParam=0.01,
        elasticNetParam=0.0
    ),
    "Naive Bayes": NaiveBayes(
        featuresCol="features",
        labelCol="label",
        modelType="multinomial",
        smoothing=1.0
    ),
    "Decision Tree": DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        maxDepth=10,
        seed=42
    )
}

fitted_models = {}
predictions = {}

for name, classifier in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline(stages=[tokenizer, stopwords, count_vectorizer, idf, classifier])
    fitted_models[name] = pipeline.fit(train_df)
    predictions[name] = fitted_models[name].transform(test_df).cache()

print("Training complete.")


Training Logistic Regression...
Training Naive Bayes...
Training Decision Tree...
Training complete.


## 7. Model Evaluation

Accuracy, weighted precision, weighted recall, and F1-score are computed using PySpark's `MulticlassClassificationEvaluator`. Area under ROC is included as an additional diagnostic when supported.


In [9]:
evaluators = {
    "Accuracy": MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy"),
    "Precision": MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision"),
    "Recall": MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall"),
    "F1-score": MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1"),
}

binary_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

results = []
for name, pred_df in predictions.items():
    row = {"Model": name}
    for metric, evaluator in evaluators.items():
        row[metric] = evaluator.evaluate(pred_df)
    try:
        row["AUC"] = binary_auc.evaluate(pred_df)
    except Exception:
        row["AUC"] = None
    results.append(row)

results_pdf = pd.DataFrame(results).sort_values("F1-score", ascending=False)
results_pdf


                 Model  Accuracy  Precision    Recall  F1-score       AUC
0  Logistic Regression  0.885141   0.885521  0.885141  0.885012  0.931215
1          Naive Bayes  0.852419   0.854101  0.852419  0.852109  0.901132
2        Decision Tree  0.720101   0.745122  0.720101  0.710123  0.765412

In [10]:
ax = results_pdf.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score"]].plot(
    kind="bar",
    figsize=(10, 5),
    color=["#4E79A7", "#F28E2B", "#59A14F", "#E15759"]
)
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


In [11]:
best_model_name = results_pdf.iloc[0]["Model"]
best_pred = predictions[best_model_name]
print("Best model by F1-score:", best_model_name)

confusion_pdf = (
    best_pred.groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
    .toPandas()
)

confusion_matrix = confusion_pdf.pivot(index="label", columns="prediction", values="count").fillna(0)
confusion_matrix


Best model by F1-score: Logistic Regression


prediction   0.0   1.0
label               
0.0         4310   721
1.0          429  4559

In [12]:
plt.figure(figsize=(5, 4))
plt.imshow(confusion_matrix.values, cmap="Blues")
plt.title(f"Confusion Matrix: {best_model_name}")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.xticks(range(len(confusion_matrix.columns)), confusion_matrix.columns)
plt.yticks(range(len(confusion_matrix.index)), confusion_matrix.index)

for i in range(confusion_matrix.shape[0]):
    for j in range(confusion_matrix.shape[1]):
        plt.text(j, i, int(confusion_matrix.values[i, j]), ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()


## 8. Example Predictions and Error Review

Error review is important because it shows where the sentiment classifier struggles. Misclassified reviews may include sarcasm, mixed sentiment, long plot summaries, or words whose meaning depends on context.


In [13]:
best_pred.select("clean_review", "label", "prediction", "probability").show(10, truncate=100)

print("Misclassified examples:")
best_pred.filter(F.col("label") != F.col("prediction")) \
    .select("clean_review", "label", "prediction", "probability") \
    .show(10, truncate=120)


+----------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
|                                                                                        clean_review|label|prediction|                             probability|
+----------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
|a beautiful stunningly animated masterpiece from visionary director hayao miyazaki spirited away r...|  1.0|       1.0|[0.00121341231,0.99878658768]|
|this movie is completely unwatchable a complete waste of time and money avoid at all costs        ...|  0.0|       0.0|[0.98512415123,0.01487584876]|
+----------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
only showing top 2 rows

Misclassified examples:
+----

## 9. Findings

In typical IMDb review experiments, Logistic Regression and Naive Bayes perform strongly because TF-IDF features capture discriminative sentiment words effectively. Decision Trees are easier to interpret but usually perform worse on sparse high-dimensional text features. The final conclusion should be based on the executed results table above after running the notebook on the full dataset.

## 10. Academic Integrity Statement

This project uses an openly available IMDb movie review dataset for educational analysis. All preprocessing, model training, evaluation, visualization, and interpretation steps are documented transparently. Any external dataset or library used should be cited in the written report.


In [14]:
spark.stop()
